# Edit Wikidata using wikibaseintegrator

In [29]:
!pip install --upgrade wikibaseintegrator

In [30]:
from wikibaseintegrator import WikibaseIntegrator
from wikibaseintegrator import wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator import datatypes
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = 'WikidataLiteraryWorksMetaDataUpload [1.0]'
USERNAME = 'Katdav-wd-lit'
    

logging.basicConfig(filename='wikibaseint-debug.log', 
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://www.wikidata.org/w/api.php'

PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'language':'P407',
         'publication_date':'P577'}
ENTITIES = {
   'literary_work':'Q7725634', 
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

In [31]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'WikidataLiteraryWorksMetaDataUpload: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

### Other possible way of logging in? Doesn't work:

In [ ]:
oauth = wbi_login.OAuth2(consumer_token=CONSUMER_TOKEN, 
                                  consumer_secret=CONSUMER_SECRET)
wbi_oauth = WikibaseIntegrator(login=oauth)


Get this entity to test:  https://www.wikidata.org/wiki/Q105624761

## Add a claim

What titles there are now?

In [33]:
flametti = wbi.item.get(entity_id='Q105624761')

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

[{'text': 'Flametti oder Vom Dandysmus der Armen', 'language': 'de'},
 {'text': 'Flametti o El dandismo de los', 'language': 'es'},
 {'text': 'Flametti o del dandismo dei poveri', 'language': 'it'},
 {'text': 'Flammetti', 'language': 'de'}]

Now we add this title to our local python representation of this entity

In [34]:
title_en_string = datatypes.MonolingualText(text='Flametti, or The Dandyism of the Poor', language='en', prop_nr=PROPS['title'])
title_en_string

flametti.claims.add(title_en_string)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

[{'text': 'Flametti oder Vom Dandysmus der Armen', 'language': 'de'},
 {'text': 'Flametti o El dandismo de los', 'language': 'es'},
 {'text': 'Flametti o del dandismo dei poveri', 'language': 'it'},
 {'text': 'Flammetti', 'language': 'de'},
 {'text': 'Flametti, or The Dandyism of the Poor', 'language': 'en'}]

## After writing there's only one title left:

In [35]:
flametti = flametti.write(login=login_instance)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

[{'text': 'Flametti, or The Dandyism of the Poor', 'language': 'en'}]

See:  https://www.wikidata.org/wiki/Q105624761

## This works:

In [ ]:
flametti.claims.add(title_en_string, action_if_exists=ActionIfExists.KEEP)